In [1]:
import os
import json
import random
from collections import defaultdict, Counter

In [2]:
!git clone https://github.com/RUCAIBox/POPE.git
%cd POPE
!python main.py \
    --seg_path ./segmentation/coco_ground_truth_segmentation.json \
    --sample_num 3 --img_num 500 --dataset coco \
    --save_path /kaggle/working/pope_output/
%cd ..

Cloning into 'POPE'...
remote: Enumerating objects: 332, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 332 (delta 12), reused 11 (delta 10), pack-reused 310 (from 1)
Receiving objects: 100% (332/332), 25.41 MiB | 18.38 MiB/s, done.
Resolving deltas: 100% (90/90), done.
/kaggle/working/POPE
/kaggle/working


In [3]:
COCO_BASE = "/kaggle/input/datasets/nadaibrahim/coco2014"

def find_file(base, filename):
    for root, _, files in os.walk(base):
        if filename in files:
            return os.path.join(root, filename)
    return None

ann_path = find_file(COCO_BASE, "instances_val2014.json")
img_dir = os.path.dirname(find_file(COCO_BASE, "COCO_val2014_000000000139.jpg") or "")

with open(ann_path) as f:
    coco = json.load(f)

cat_id_to_name = {c["id"]: c["name"] for c in coco["categories"]}

image_to_objects = defaultdict(set)
for ann in coco["annotations"]:
    image_to_objects[ann["image_id"]].add(cat_id_to_name[ann["category_id"]])

image_id_to_filename = {img["id"]: img["file_name"] for img in coco["images"]}
coco_categories = set(cat_id_to_name.values())

print(len(image_to_objects), "images with ground-truth objects,", len(coco_categories), "categories")

40137 images with ground-truth objects, 80 categories


In [4]:
!git clone https://github.com/nickjiang2378/vlm-hallucinations.git

import re

with open("vlm-hallucinations/metric/chair.py") as f:
    content = f.read()

match = re.search(r"synonyms_txt = '''(.*?)'''", content, re.DOTALL)
synonyms_txt = match.group(1)


def build_synonym_dict(synonyms_txt: str) -> dict:
    synonym_dict = {}
    for line in synonyms_txt.strip().split("\n"):
        words = [w.strip() for w in line.split(",") if w.strip()]
        if not words:
            continue
        canonical = words[0]
        for w in words:
            synonym_dict[w] = canonical
    return synonym_dict


synonym_dict = build_synonym_dict(synonyms_txt)
print(len(synonym_dict), "synonym entries")

Cloning into 'vlm-hallucinations'...
remote: Enumerating objects: 386, done.
remote: Counting objects: 100% (101/101), done.
remote: Compressing objects: 100% (59/59), done.
remote: Total 386 (delta 51), reused 42 (delta 42), pack-reused 285 (from 1)
Receiving objects: 100% (386/386), 24.12 MiB | 24.08 MiB/s, done.
Resolving deltas: 100% (151/151), done.
403 synonym entries


In [5]:
import nltk
nltk.download('punkt_tab')
nltk.download('wordnet')

[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [6]:
from nltk import word_tokenize
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()


def extract_coco_objects_direct(caption: str, synonym_dict: dict, coco_categories: set) -> set:
    tokens = word_tokenize(caption.lower())
    lemmatized = [lemmatizer.lemmatize(t) for t in tokens]
    mapped = set()
    for tok in lemmatized:
        if tok in coco_categories:
            mapped.add(tok)
        elif tok in synonym_dict:
            mapped.add(synonym_dict[tok])
    return mapped


def label_caption(image_id: int, caption: str, image_to_objects: dict, synonym_dict: dict, coco_categories: set) -> list:
    mentioned = extract_coco_objects_direct(caption, synonym_dict, coco_categories)
    ground_truth = image_to_objects.get(image_id, set())
    return [
        {"image_id": image_id, "object": obj, "real": obj in ground_truth}
        for obj in mentioned
    ]


test_caption = "A dog sits next to a wooden bench in a park. A frisbee lies nearby."
sample_id = coco["images"][0]["id"]
for row in label_caption(sample_id, test_caption, image_to_objects, synonym_dict, coco_categories):
    print(row)

{'image_id': 391895, 'object': 'dog', 'real': False}
{'image_id': 391895, 'object': 'frisbee', 'real': False}
{'image_id': 391895, 'object': 'bench', 'real': False}


In [7]:
def label_pope_questions(path: str) -> list:
    """POPE's own yes/no questions, in the unified question-list schema. This is the
    PRIMARY object-hallucination baseline -- discriminative, same reasoning as why AMBER's
    and Reefknot's discriminative tasks are primary for attribute/relation."""
    rows = []
    with open(path) as f:
        for i, line in enumerate(f):
            d = json.loads(line)
            rows.append({
                "image_id": os.path.splitext(d["image"])[0],
                "question_id": f"pope_{i}",
                "question_text": d["text"],
                "ground_truth_answer": d["label"],  # "yes" / "no"
                "answer_format": "yes_no",
                "hallucination_type": "object",
            })
    return rows


pope_questions = label_pope_questions("/kaggle/working/pope_output/coco/coco_pope_random.json")
print(len(pope_questions), "POPE questions loaded")
print(pope_questions[0])

3000 POPE questions loaded
{'image_id': 'COCO_val2014_000000060623', 'question_id': 'pope_0', 'question_text': 'Is there a bowl in the image?', 'ground_truth_answer': 'yes', 'answer_format': 'yes_no', 'hallucination_type': 'object'}


In [8]:
!git clone https://github.com/junyangwang0410/AMBER.git

with open("AMBER/data/query/query_discriminative-attribute.json") as f:
    amber_attribute_q = json.load(f)
with open("AMBER/data/query/query_discriminative-relation.json") as f:
    amber_relation_q = json.load(f)
with open("AMBER/data/annotations.json") as f:
    amber_ann = json.load(f)

amber_ann_by_id = {a["id"]: a for a in amber_ann}


def build_amber_questions(query_list: list, hallucination_type: str) -> list:
    rows = []
    for q in query_list:
        ann = amber_ann_by_id[q["id"]]
        rows.append({
            "image_id": os.path.splitext(q["image"])[0],
            "question_id": f"amber_{q['id']}",
            "question_text": q["query"],
            "ground_truth_answer": ann["truth"],
            "answer_format": "yes_no",
            "hallucination_type": hallucination_type,
        })
    return rows


amber_questions = (
    build_amber_questions(amber_attribute_q, "attribute")
    + build_amber_questions(amber_relation_q, "relation")
)
print(len(amber_questions), "AMBER attribute+relation questions")
print(amber_questions[0])

Cloning into 'AMBER'...
remote: Enumerating objects: 41, done.
remote: Counting objects: 100% (41/41), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 41 (delta 16), reused 21 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (41/41), 874.48 KiB | 4.33 MiB/s, done.
Resolving deltas: 100% (16/16), done.
9292 AMBER attribute+relation questions
{'image_id': 'AMBER_1', 'question_id': 'amber_1005', 'question_text': 'Is the sky sunny in this image?', 'ground_truth_answer': 'yes', 'answer_format': 'yes_no', 'hallucination_type': 'attribute'}


In [9]:
AMBER_IMAGE_DIR = "/kaggle/input/datasets/nocturnalnerd18/amber-hallucination/image"

In [10]:
"""
!wget https://cs.stanford.edu/people/rak248/VG_100K/images.zip -O /kaggle/working/VG_100K.zip
!wget https://cs.stanford.edu/people/rak248/VG_100K_2/images2.zip -O /kaggle/working/VG_100K_2.zip
"""
!unzip -q -o /kaggle/working/VG_100K.zip -d /kaggle/temp/
!unzip -q -o /kaggle/working/VG_100K_2.zip -d /kaggle/temp/

unzip:  cannot find or open /kaggle/working/VG_100K.zip, /kaggle/working/VG_100K.zip.zip or /kaggle/working/VG_100K.zip.ZIP.
unzip:  cannot find or open /kaggle/working/VG_100K_2.zip, /kaggle/working/VG_100K_2.zip.zip or /kaggle/working/VG_100K_2.zip.ZIP.


In [11]:
!git clone https://github.com/JackChen-seu/Reefknot.git

def load_jsonl(path):
    rows = []
    with open(path) as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

reefknot_yesno = load_jsonl("Reefknot/Dataset/YESNO.jsonl")
reefknot_mcq = load_jsonl("Reefknot/Dataset/Multichoice.jsonl")
reefknot_all = reefknot_yesno + reefknot_mcq

reefknot_by_image = defaultdict(list)
for r in reefknot_all:
    reefknot_by_image[r["image_id"]].append(r)

print(len(reefknot_by_image), "unique images across YESNO+MCQ,", len(reefknot_all), "questions total")

Cloning into 'Reefknot'...
remote: Enumerating objects: 3940, done.
remote: Counting objects: 100% (282/282), done.
remote: Compressing objects: 100% (162/162), done.
remote: Total 3940 (delta 125), reused 256 (delta 108), pack-reused 3658 (from 1)
Receiving objects: 100% (3940/3940), 36.65 MiB | 19.25 MiB/s, done.
Resolving deltas: 100% (1083/1083), done.
8827 unique images across YESNO+MCQ, 16690 questions total


In [12]:
def stratified_image_sample(by_image: dict, target_total: int, seed: int = 42) -> list:
    """Sample images (not questions) preserving the perception:cognitive ratio, using
    each image's majority relation_type as its bucket."""
    image_majority = {}
    for img_id, qs in by_image.items():
        types = Counter(q["relation_type"] for q in qs)
        image_majority[img_id] = types.most_common(1)[0][0]

    perception_imgs = [i for i, t in image_majority.items() if t == "perception"]
    cognitive_imgs = [i for i, t in image_majority.items() if t == "cognitive"]

    perception_frac = len(perception_imgs) / len(by_image)
    n_perception = round(target_total * perception_frac)
    n_cognitive = target_total - n_perception

    random.seed(seed)
    return random.sample(perception_imgs, n_perception) + random.sample(cognitive_imgs, n_cognitive)


REEFKNOT_TARGET_IMAGES = 500
reefknot_sampled_images = set(stratified_image_sample(reefknot_by_image, REEFKNOT_TARGET_IMAGES))
print(len(reefknot_sampled_images), "images sampled")
print(sum(len(reefknot_by_image[i]) for i in reefknot_sampled_images), "questions in the sample")

500 images sampled
938 questions in the sample


In [13]:
def build_reefknot_questions(by_image: dict, sampled_images: set) -> list:
    rows = []
    for img_id in sampled_images:
        for i, q in enumerate(by_image[img_id]):
            is_mcq = q["type"] == "Multichoice"
            rows.append({
                "image_id": img_id,
                "question_id": f"reefknot_{img_id}_{i}",
                "question_text": q["query_prompt"],
                "ground_truth_answer": q["label"],
                "answer_format": "mcq" if is_mcq else "yes_no",
                "hallucination_type": "relation",
                "relation_type": q["relation_type"],  # perception / cognitive -- kept for later breakdowns
            })
    return rows


reefknot_questions = build_reefknot_questions(reefknot_by_image, reefknot_sampled_images)
print(len(reefknot_questions), "Reefknot questions")
print(reefknot_questions[0])

938 Reefknot questions
{'image_id': '2399525', 'question_id': 'reefknot_2399525_0', 'question_text': 'Is the plate before child in this photo? Please answer yes or no.', 'ground_truth_answer': 'yes', 'answer_format': 'yes_no', 'hallucination_type': 'relation', 'relation_type': 'perception'}


In [14]:
VG_IMAGE_DIRS = ["/kaggle/temp", "/kaggle/temp/VG_100K_2"]

def find_vg_image_path(image_id: str) -> str:
    for d in VG_IMAGE_DIRS:
        candidate = os.path.join(d, f"{image_id}.jpg")
        if os.path.exists(candidate):
            return candidate
    return None

In [15]:
def make_split_manifest(image_ids: list, train=0.7, val=0.15, seed: int = 42) -> dict:
    ids = list(image_ids)
    random.seed(seed)
    random.shuffle(ids)
    n = len(ids)
    n_train = int(n * train)
    n_val = int(n * val)
    manifest = {}
    for i, img_id in enumerate(ids):
        if i < n_train:
            manifest[img_id] = "train"
        elif i < n_train + n_val:
            manifest[img_id] = "val"
        else:
            manifest[img_id] = "test"
    return manifest


pope_image_ids = sorted({q["image_id"] for q in pope_questions})
amber_image_ids = sorted({q["image_id"] for q in amber_questions})
reefknot_image_ids = sorted(reefknot_sampled_images)

manifests = {
    "pope_split.json": make_split_manifest(pope_image_ids),
    "amber_split.json": make_split_manifest(amber_image_ids),
    "reefknot_split.json": make_split_manifest(reefknot_image_ids),
}

os.makedirs("/kaggle/working/splits", exist_ok=True)
for fname, manifest in manifests.items():
    out_path = os.path.join("/kaggle/working/splits", fname)
    with open(out_path, "w") as f:
        json.dump(manifest, f)
    counts = Counter(manifest.values())
    print(fname, dict(counts))

pope_split.json {'train': 350, 'val': 75, 'test': 75}
amber_split.json {'train': 702, 'val': 150, 'test': 152}
reefknot_split.json {'train': 350, 'val': 75, 'test': 75}


In [16]:
os.makedirs("/kaggle/working/questions", exist_ok=True)

all_questions = pope_questions + amber_questions + reefknot_questions

with open("/kaggle/working/questions/pope_questions.json", "w") as f:
    json.dump(pope_questions, f)
with open("/kaggle/working/questions/amber_questions.json", "w") as f:
    json.dump(amber_questions, f)
with open("/kaggle/working/questions/reefknot_questions.json", "w") as f:
    json.dump(reefknot_questions, f)
with open("/kaggle/working/questions/all_questions.json", "w") as f:
    json.dump(all_questions, f)

print(len(all_questions), "total discriminative questions across all three datasets:")
print(Counter(q["hallucination_type"] for q in all_questions))

!kaggle datasets init -p /kaggle/working/questions/
!kaggle datasets init -p /kaggle/working/splits/

13230 total discriminative questions across all three datasets:
Counter({'attribute': 7628, 'object': 3000, 'relation': 2602})
Data package template written to: /kaggle/working/questions/dataset-metadata.json
Data package template written to: /kaggle/working/splits/dataset-metadata.json


In [17]:
!kaggle datasets create -p /kaggle/working/questions/
!kaggle datasets create -p /kaggle/working/splits/

Default slug detected, please change values before uploading
Default slug detected, please change values before uploading


In [18]:
import re

def normalize_answer(answer_text: str, answer_format: str):
    text = answer_text.strip().lower()

    if answer_format == "yes_no":
        m = re.match(r"^\W*(yes|no)\b", text)
        if m:
            return m.group(1)
        has_yes = re.search(r"\byes\b", text) is not None
        has_no = re.search(r"\bno\b", text) is not None
        if has_yes and not has_no:
            return "yes"
        if has_no and not has_yes:
            return "no"
        return None

    if answer_format == "mcq":
        m = re.match(r"^\(?([abcd])(?:[\).:]|$)", text)
        return m.group(1) if m else None

    return None

In [19]:
import glob
import torch

def label_from_model_answers(cache_dirs: list):
    labeled, unparseable = [], []
    for cache_dir in cache_dirs:
        for batch_path in glob.glob(os.path.join(cache_dir, "*.pt")):
            batch = torch.load(batch_path)
            for entry in batch:
                model_answer = normalize_answer(entry["answer_text"], entry["answer_format"])
                if model_answer is None:
                    unparseable.append((entry["question_id"], entry["answer_format"], entry["answer_text"]))
                    continue
                labeled.append({
                    "image_id": entry["image_id"],
                    "question_id": entry["question_id"],
                    "hallucination_type": entry["hallucination_type"],
                    "answer_format": entry["answer_format"],
                    "ground_truth_answer": entry["ground_truth_answer"],
                    "model_answer": model_answer,
                    "answer_text": entry["answer_text"],
                    "correct": model_answer == entry["ground_truth_answer"].strip().lower(),
                })
    print(f"{len(labeled)} labeled, {len(unparseable)} unparseable")
    return labeled, unparseable

In [20]:
labeled, unparseable = label_from_model_answers([
    "/kaggle/input/datasets/nocturnalnerd18/discriminative-cache-pope",
    "/kaggle/input/datasets/nocturnalnerd18/discriminative-cache-amber",
    "/kaggle/input/datasets/nocturnalnerd18/discriminative-cache-reefknot",
])

13222 labeled, 8 unparseable


In [21]:
import pandas as pd
df = pd.DataFrame(labeled)
df["source"] = df["question_id"].str.split("_").str[0]

print(pd.crosstab([df.hallucination_type, df.source, df.answer_format], df.correct))
print("\nunparseable by format:\n", pd.Series([u[1] for u in unparseable]).value_counts())
for u in unparseable[:15]:
    print(u)

correct                                    False  True 
hallucination_type source   answer_format              
attribute          amber    yes_no          2025   5595
object             pope     yes_no           319   2681
relation           amber    yes_no           470   1194
                   reefknot mcq              214    144
                            yes_no           213    367

unparseable by format:
 yes_no    8
Name: count, dtype: int64
('amber_6223', 'yes_no', 'There are three oranges')
('amber_6224', 'yes_no', 'There are four oranges')
('amber_6243', 'yes_no', 'There are two oranges')
('amber_6244', 'yes_no', 'There are two oranges')
('amber_2534', 'yes_no', 'There are six oranges')
('amber_4707', 'yes_no', 'There are three oranges')
('amber_4708', 'yes_no', 'There are six oranges')
('amber_6012', 'yes_no', 'In the image, the')


In [22]:
from collections import Counter
totals = Counter(l["hallucination_type"] for l in labeled)
wrong = Counter(l["hallucination_type"] for l in labeled if not l["correct"])

for t in totals:
    rate = wrong[t] / totals[t] * 100
    print(f"{t}: {wrong[t]}/{totals[t]} hallucinated ({rate:.1f}%)")

object: 319/3000 hallucinated (10.6%)
attribute: 2025/7620 hallucinated (26.6%)
relation: 897/2602 hallucinated (34.5%)


In [23]:
os.makedirs("/kaggle/working/labels", exist_ok=True)
with open("/kaggle/working/labels/labeled_examples.json", "w") as f:
    json.dump(labeled, f)

In [24]:
import json

with open("/kaggle/working/labels/dataset-metadata.json", "w") as f:
    json.dump({
        "title": "vlm-labeled-examples",
        "id": "nocturnalnerd18/vlm-labeled-examples",
        "licenses": [{"name": "CC0-1.0"}]
    }, f)

!kaggle datasets version -p /kaggle/working/labels/ -m "Fix MCQ case bug; add answer_format, model_answer, answer_text"

Starting upload for file labeled_examples.json
100%|██████████████████████████████████████| 2.83M/2.83M [00:00<00:00, 6.63MB/s]
Upload successful: labeled_examples.json (3MB)
Dataset version is being created. Please check progress at https://www.kaggle.com/datasets/nocturnalnerd18/vlm-labeled-examples


In [31]:
import shutil, glob

Q_IN, S_IN = "/kaggle/input/datasets/nocturnalnerd18/vlm-questions", "/kaggle/input/datasets/nocturnalnerd18/vlm-splits"
Q_OUT, S_OUT = "/kaggle/working/questions_v2", "/kaggle/working/splits_v2"
for d in (Q_OUT, S_OUT):
    shutil.rmtree(d, ignore_errors=True)
    os.makedirs(d)

# start from the published files so nothing else in the dataset gets dropped
for src_dir, out_dir in ((Q_IN, Q_OUT), (S_IN, S_OUT)):
    for path in glob.glob(f"{src_dir}/*.json"):
        if not path.endswith("dataset-metadata.json"):
            shutil.copy(path, out_dir)
print("questions files:", sorted(os.listdir(Q_OUT)))
print("splits files:", sorted(os.listdir(S_OUT)))

# append the new Reefknot questions
with open(f"{Q_OUT}/all_questions.json") as f:
    all_q = json.load(f)
assert len(all_q) == 13230, len(all_q)
assert not ({q["question_id"] for q in all_q} & {q["question_id"] for q in extra_questions}), "id collision"
all_q_v2 = all_q + extra_questions
with open(f"{Q_OUT}/all_questions.json", "w") as f:
    json.dump(all_q_v2, f)

p = f"{Q_OUT}/reefknot_questions.json"      # keep the per-source file in step, if it exists
if os.path.exists(p):
    with open(p) as f:
        rk = json.load(f)
    with open(p, "w") as f:
        json.dump(rk + extra_questions, f)

# add the new images to the Reefknot split only
with open(f"{S_OUT}/reefknot_split.json") as f:
    rk_split = json.load(f)
assert len(rk_split) == 500, len(rk_split)
assert not (set(rk_split) & set(extra_split)), "image overlap with existing split"
rk_split.update(extra_split)
with open(f"{S_OUT}/reefknot_split.json", "w") as f:
    json.dump(rk_split, f)

print(len(all_q_v2), "total questions;", len(rk_split), "Reefknot images;", Counter(rk_split.values()))

questions files: ['all_questions.json', 'amber_questions.json', 'pope_questions.json', 'reefknot_questions.json']
splits files: ['amber_split.json', 'pope_split.json', 'reefknot_split.json']
17492 total questions; 2500 Reefknot images; Counter({'train': 1750, 'val': 375, 'test': 375})


In [32]:
for out_dir, slug in ((Q_OUT, "vlm-questions"), (S_OUT, "vlm-splits")):
    with open(f"{out_dir}/dataset-metadata.json", "w") as f:
        json.dump({"title": slug, "id": f"nocturnalnerd18/{slug}", "licenses": [{"name": "CC0-1.0"}]}, f)

!kaggle datasets version -p /kaggle/working/questions_v2 -m "Add 4,262 Reefknot yes/no questions on 2,000 new images"
!kaggle datasets version -p /kaggle/working/splits_v2 -m "Add split assignments for 2,000 new Reefknot images"

Starting upload for file pope_questions.json
100%|████████████████████████████████████████| 621k/621k [00:00<00:00, 1.72MB/s]
Upload successful: pope_questions.json (621KB)
Starting upload for file all_questions.json
100%|██████████████████████████████████████| 3.79M/3.79M [00:00<00:00, 9.73MB/s]
Upload successful: all_questions.json (4MB)
Starting upload for file amber_questions.json
100%|██████████████████████████████████████| 1.82M/1.82M [00:00<00:00, 5.15MB/s]
Upload successful: amber_questions.json (2MB)
Starting upload for file reefknot_questions.json
100%|██████████████████████████████████████| 1.36M/1.36M [00:00<00:00, 3.73MB/s]
Upload successful: reefknot_questions.json (1MB)
Dataset version is being created. Please check progress at https://www.kaggle.com/datasets/nocturnalnerd18/vlm-questions
Starting upload for file reefknot_split.json
100%|███████████████████████████████████████| 47.5k/47.5k [00:00<00:00, 131kB/s]
Upload successful: reefknot_split.json (47KB)
Starting uplo